# 🔬 Notebook 2: Gates & Measurement

### *Ode to Quantum: Meridian Station Quantum Core Lab*

> Change the state. Then find out what changed.

---


## 🛰️ Mission Briefing


*"Cadet, a circuit that never changes anything is just decoration."* ECHO brings up a
diagnostic panel showing five glowing icons: **X, Y, Z, H, I**.

*"These are your first tools: quantum gates. Each one reshapes a qubit's state in a
specific, predictable way. But here's the catch: you can't just peek at a qubit to
see what a gate did. You have to **measure** it, and measurement only gives you a
classical answer ( 0 or 1 ) drawn from an underlying probability. Run the experiment
once, and you'll learn almost nothing. Run it a thousand times, and the truth
emerges."*


## 🎯 Learning Objectives

By the end of this notebook, you will be able to:

- Apply the five foundational single-qubit gates: X, Y, Z, H, and I
- Predict and verify each gate's effect using the Bloch sphere
- Read each gate's matrix representation
- Measure a qubit and interpret the result as a classical bit
- Run many shots and read out results as probabilities using a histogram
- Explain why the number of shots affects how trustworthy your results are


## 🧩 Prerequisites

- Notebook 0: building and drawing a `QuantumCircuit`
- Notebook 1: reading a `Statevector` and ket notation `|0⟩`, `|1⟩`


## 💡 Concept


**The five gates you'll use constantly:**

| Gate | Nickname | What it does |
|---|---|---|
| `I` | Identity | Does nothing. Useful as a placeholder and a sanity check. |
| `X` | Bit-flip | Swaps `|0⟩` ↔ `|1⟩`. The quantum version of a classical NOT. |
| `Z` | Phase-flip | Leaves `|0⟩` alone, flips the *sign* of `|1⟩`. Invisible if you only look at probabilities — but not invisible to other gates. |
| `Y` | Bit + phase flip | Does both of the above, together, with a twist (literally — it involves imaginary numbers). |
| `H` | Hadamard | Takes a qubit that is definitely `0` or definitely `1` and pushes it into **superposition**; equal parts `0` and `1`. This is the single most important gate in this entire course. |

**Measurement.**

A quantum circuit doesn't hand you a state vector directly. Real hardware can't
"read" a qubit like a variable in code. Instead, you **measure** it, and the qubit
collapses to a classical outcome: `0` or `1`. If you run the exact same circuit many
times ("shots"), the *proportion* of 0s and 1s approximates the underlying
probabilities. This is why quantum programs are run in batches of shots rather than
just once.


## 📊 Visualization


The Bloch sphere is a way to visualize a single qubit's state as a point on (or in)
a 3D sphere. `|0⟩` sits at the north pole, `|1⟩` at the south pole. Let's watch each
gate move that point.


You likely need to install Qiskit and other libraries first in a terminal, run:

In [ ]:
%pip install qiskit qiskit-aer matplotlib pylatexenc qiskit[quantum_info] qiskit[visualisation]

## 🧪 Hands-on Code


First, a small helper so you can visualize any gate's effect in one line.


In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, Operator
from qiskit.visualization import plot_bloch_multivector
from qiskit_aer import AerSimulator
from qiskit import transpile
import matplotlib.pyplot as plt

def bloch_after(gate_name, starting_state="0"):
    """Apply one gate to a qubit (optionally starting from |1>) and plot the Bloch sphere."""
    qc = QuantumCircuit(1)
    if starting_state == "1":
        qc.x(0)  # flip to |1> first, then apply the gate we're studying
    if gate_name != "id":
        getattr(qc, gate_name)(0)
    return plot_bloch_multivector(Statevector(qc), title=f"After {gate_name.upper()} (from |{starting_state}⟩)")

bloch_after("x")


In [ ]:
bloch_after("h")


In [ ]:
bloch_after("z", starting_state="1")


Notice: `X` swings the point from the north pole to the south pole (a full flip).
`H` sends it sideways onto the equator - neither pole, a genuine superposition.
`Z` applied to `|1⟩` doesn't move the point on the sphere at all in this view (`Z`'s
effect is a phase, which the sphere still encodes as orientation around the vertical
axis - subtle, but real, as you'll see when `Z` is combined with `H` later in the
course).

## 📐 Math Lens


Every single-qubit gate is a $2\times2$ matrix. Applying the gate means multiplying
that matrix by the qubit's state vector.

$$
X = \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix}
\qquad
Y = \begin{pmatrix} 0 & -i \\ i & 0 \end{pmatrix}
\qquad
Z = \begin{pmatrix} 1 & 0 \\ 0 & -1 \end{pmatrix}
\qquad
H = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 & 1 \\ 1 & -1 \end{pmatrix}
$$

You don't need to hand-multiply these to use Qiskit but it's worth seeing that
Qiskit isn't hiding anything. Ask it directly for the matrix behind any gate:

In [ ]:
for gate in ["x", "y", "z", "h", "id"]:
    qc = QuantumCircuit(1)
    getattr(qc, gate)(0)
    print(gate.upper(), "=")
    print(Operator(qc).data.round(3))
    print()


## 🔁 Experiments


Now let's measure. Build a circuit, apply a gate, add a measurement, and run it on a
simulator for a chosen number of shots.


In [ ]:
from qiskit.visualization import plot_histogram

def run_and_plot(gate_name, shots):
    qc = QuantumCircuit(1, 1)
    getattr(qc, gate_name)(0)
    qc.measure(0, 0)

    backend = AerSimulator()
    tqc = transpile(qc, backend)
    counts = backend.run(tqc, shots=shots).result().get_counts()
    return counts, plot_histogram(counts, title=f"{gate_name.upper()} — {shots} shots")

counts_100, fig_100 = run_and_plot("h", 100)
print(counts_100)
fig_100


In [ ]:
counts_500, fig_500 = run_and_plot("h", 500)
print(counts_500)
fig_500


In [ ]:
counts_1000, fig_1000 = run_and_plot("h", 1000)
print(counts_1000)
fig_1000


Look at the *ratio* of 0s to 1s, not the raw counts, as the shot count climbs from
100 → 500 → 1000. It should keep drifting closer to a clean 50/50 split. That's the
law of large numbers at work. The same reason a fair coin flipped 10 times might
give you 7 heads, but flipped 10,000 times will land much closer to 50%.


## 🚀 Challenge


**Predict before you run.**

1. What do you expect if you apply `H` **twice** in a row, then measure? Write down
   a one-sentence prediction.
2. Now build it: one qubit, `qc.h(0)` twice, then measure, then run it for 1000
   shots and plot the histogram.
3. Was your prediction right? If not, what does that tell you about what `H` is
   actually doing: is it random, or is it a precise, reversible operation that
   just *looks* random after one application?


In [ ]:
# Your turn — apply H twice, measure, run 1000 shots, plot the histogram.


## 🪞 Reflection


`H` applied once looks like a coin flip. `H` applied twice is perfectly
deterministic - you get back exactly what you started with, every time. A single
measurement can make a quantum operation *look* random, but the operation itself is
not random at all: it's a precise rotation. Randomness only enters at the moment of
measurement. Keep this distinction in mind - it will matter again the moment you
meet interference in later notebooks.

## 📦 Summary

- X, Y, Z are flip-style gates; H creates superposition; I does nothing
- Every single-qubit gate is a 2×2 matrix; Qiskit's `Operator` can show it to you directly
- Measurement collapses a qubit to a classical 0 or 1, drawn from a probability distribution
- More shots → the observed counts converge more tightly to the true probabilities
- A gate can look random after one run and be perfectly deterministic in reverse: HH = I


## ➡️ Next Mission


You've seen `H` push a qubit onto the equator of the Bloch sphere and called it
"superposition" in passing. In **Notebook 3: Superposition**, you'll stop taking
that word for granted: you'll compute exact probabilities from amplitudes, sweep a
continuous rotation instead of a fixed gate, and predict measurement outcomes before
you ever press run.


---
*End of transmission.*
